# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ABK998/flyrank-ML/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why


Method Selected: Random Forest Classifier.

Why it fits: Search traffic decay involves non-linear conditional interactions (e.g., a position of 5 behaves differently for an article with 50,000 impressions versus an isolated page). Random Forest handles non-linear feature interactions natively, requires minimal preprocessing, and is robust against extreme traffic outliers.

In [3]:
# Signal inspection and feature preparation
df["target_is_declining"] = (df["trend_direction"] == "down").astype(int)
features = ["impressions_90d", "avg_position", "content_age_days", "ctr", "word_count"]
X = df[features].fillna(0)
y = df["target_is_declining"]
print(f"Feature matrix shape: {X.shape}, Target distribution: {y.mean():.2%} declining")

Feature matrix shape: (30000, 5), Target distribution: 54.21% declining


## 2. Split design

Split Strategy: Group-shuffle split grouped by client_id (80/20 train/test split).

Why it is honest: Standard random splits cause domain leakage by placing different pages from the same website into both training and testing sets. Grouping by client ensures all pages from a specific domain stay strictly on one side, forcing the model to generalize across unseen websites rather than memorizing domain structure.

In [4]:
groups = df["client_id"]
gss = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups))

X_train, y_train = X.iloc[train_idx], y.iloc[train_idx]
X_test, y_test = X.iloc[test_idx], y.iloc[test_idx]

print(f"Training split: {len(X_train):,} rows ({df.iloc[train_idx]['client_id'].nunique()} clients)")
print(f"Testing split:  {len(X_test):,} rows ({df.iloc[test_idx]['client_id'].nunique()} clients)")

Training split: 23,837 rows (25 clients)
Testing split:  6,163 rows (7 clients)


## 3. Train + compare vs my baseline
*Same data, same metric, same split as your Week-4 baseline. Show the table.*

Evaluation Metric: Precision@50 (matching Week-4 baseline). Evaluated on the exact same unseen test split.

In [5]:
# 1. Replicate Week-4 baseline on test set
test_df = df.iloc[test_idx].copy()
max_imp = test_df["impressions_90d"].quantile(0.95)
norm_imp = (test_df["impressions_90d"].clip(0, max_imp)) / max_imp
max_age = test_df["content_age_days"].quantile(0.95)
norm_age = (test_df["content_age_days"].clip(0, max_age)) / max_age
pos_opp = np.where(test_df["avg_position"].between(4.0, 15.0), 1.0,
          np.where(test_df["avg_position"].between(1.0, 3.9), 0.5, 0.2))

test_df["baseline_score"] = (0.45 * norm_imp + 0.35 * norm_age + 0.20 * pos_opp) * 100.0

# 2. Train Random Forest model
rf = RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)
test_df["model_score"] = rf.predict_proba(X_test)[:, 1]

# 3. Compare Precision@50
def precision_at_k(df_eval, score_col, target_col, k=50):
    top_k = df_eval.sort_values(by=score_col, ascending=False).head(k)
    return top_k[target_col].mean()

p50_base = precision_at_k(test_df, "baseline_score", "target_is_declining", k=50)
p50_model = precision_at_k(test_df, "model_score", "target_is_declining", k=50)

comparison_table = pd.DataFrame({
    "Approach": ["Week 4 Heuristic Baseline", "Random Forest Model (ML-08)"],
    "Precision@50": [p50_base, p50_model],
    "Correct Refreshes (out of 50)": [round(p50_base * 50), round(p50_model * 50)]
})
print(comparison_table.to_string(index=False))

                   Approach  Precision@50  Correct Refreshes (out of 50)
  Week 4 Heuristic Baseline           0.4                             20
Random Forest Model (ML-08)           0.6                             30


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*


Error Analysis: The model's false positives in the top 50 are heavily concentrated on evergreen, high-impression navigational pages that have high age but maintain stable traffic. The model struggles to differentiate between aging informational content (which decays) and permanent brand assets (which remain stable).

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Feature importances & False Positive inspection
importance_df = pd.DataFrame({
    "Feature": features,
    "Gini Importance": rf.feature_importances_
}).sort_values(by="Gini Importance", ascending=False)

print("Feature Importances:")
print(importance_df.to_string(index=False))

top_50_model = test_df.sort_values(by="model_score", ascending=False).head(50)
false_positives = top_50_model[top_50_model["target_is_declining"] == 0]
print(f"\nFalse Positives in Model Top 50: {len(false_positives)} pages out of 50")

Feature Importances:
         Feature  Gini Importance
 impressions_90d         0.319320
    avg_position         0.242057
content_age_days         0.208956
      word_count         0.142806
             ctr         0.086860

False Positives in Model Top 50: 20 pages out of 50


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.